In [ ]:
# Cell 1：最初はこのセルだけ実行し、入力欄を自分の値に変更してください。
# SET_ / REPLACE_WITH_ は未設定です。既存Widgetは再作成しません。
defaults = {
    "work_root": "/Volumes/REPLACE_WITH_CATALOG/REPLACE_WITH_SCHEMA/REPLACE_WITH_WORK_VOLUME",
    "comparison_run_id": "SET_COMPARISON_RUN_ID",
    "ai_run_id": "SET_NEW_AI_RUN_ID",
    "target_area_code": "13104",
    "top_n": "5"
}
labels = {
    "work_root": "成果物Volumeの絶対パス（4本で共通）",
    "comparison_run_id": "比較run ID：nb02は未使用、nb03は61件PASS済みID",
    "ai_run_id": "機械学習ID：nb03は未使用、nb04はPASS済みID",
    "target_area_code": "類似検索の対象地域コード（13104＝新宿区）",
    "top_n": "類似検索の表示件数（既定5）"
}
for name, value in defaults.items():
    try:
        aidputils.widgets.get(name)
    except Exception:
        aidputils.widgets.text(name=name, defaultValue=value, label=labels[name])
for name in defaults:
    print(name, '=', aidputils.widgets.get(name))


# nb03: 2020→2025の変化を機械学習で分類する

比較NotebookのPASSを確認し、検査済みの変化量3指標をStandardScaler＋KMeansで分類します。
各市区町村を等しい重みで扱う探索的分析です。将来予測、因果推論、地域の優劣の判定は行いません。
入力62市区町村から、比較Notebookで公式資料に基づき境域変更を除外した場合は61市区町村を対象にします。
クラスタ数はk=2〜6、seedは42・7・21で検証します。全seedで最小クラスタ2以上の候補から平均silhouette最大を選びます。
seed間の全ペアARIも記録します。妥当な候補がなければ結果を作らず停止します。

このNotebookは自分でデータにモデルを学習させる教師なし学習を実装しています。
ローカル検証とAIDP実行は別です。配布Notebookの実行結果は空欄で、AIDP上の実測ログを表していません。
依存パッケージをComputeにインストールした後、Sparkが利用可能なInteractive Notebookで上から実行します。


In [ ]:
from pathlib import Path
import json
import platform
import re
from datetime import datetime, timezone

def text_widget(name, default, label):
    try:
        aidputils.widgets.get(name)
    except Exception:
        aidputils.widgets.text(name=name, defaultValue=default, label=label)
    return aidputils.widgets.get(name)

WORK_ROOT = text_widget("work_root", "/Volumes/REPLACE_WITH_CATALOG/REPLACE_WITH_SCHEMA/REPLACE_WITH_WORK_VOLUME", "External Volume: work root").rstrip("/")
COMPARISON_RUN_ID = text_widget("comparison_run_id", "SET_COMPARISON_RUN_ID", "PASS済みの比較run_id")
AI_RUN_ID = text_widget("ai_run_id", "SET_NEW_AI_RUN_ID", "新しい機械学習run_id")
TARGET_AREA_CODE = text_widget("target_area_code", "13104", "似た変化を探す市区町村コード")
TOP_N = int(text_widget("top_n", "5", "表示する類似市区町村数"))
for name, value in (("comparison_run_id", COMPARISON_RUN_ID), ("ai_run_id", AI_RUN_ID)):
    if value.startswith("SET_") or not re.fullmatch(r"[A-Za-z0-9_-]+", value):
        raise ValueError(f"{name} に実行IDを設定して、このセルを再実行してください")
if "REPLACE_WITH_" in WORK_ROOT or not WORK_ROOT.startswith("/Volumes/") or ".." in Path(WORK_ROOT).parts:
    raise ValueError("work_root は /Volumes/ 以下の絶対パスにしてください")
if not re.fullmatch(r"\d{5}", TARGET_AREA_CODE):
    raise ValueError("target_area_code must be five digits")

COMPARISON_ROOT = f"{WORK_ROOT}/comparisons/{COMPARISON_RUN_ID}"
INPUT_PATH = f"{COMPARISON_ROOT}/curated/tokyo_changes"
OUTPUT_ROOT = Path(f"{WORK_ROOT}/ai_runs/{AI_RUN_ID}/ml")
if OUTPUT_ROOT.exists():
    raise FileExistsError(f"既存runは上書きしません。新しい ai_run_id を指定してください: {OUTPUT_ROOT}")
print(json.dumps({"comparison_run_id": COMPARISON_RUN_ID, "ai_run_id": AI_RUN_ID,
                  "input": INPUT_PATH, "output": str(OUTPUT_ROOT)}, ensure_ascii=False, indent=2))


## 学習・検査処理

次のセルは `ml_change_core.py` と同じ処理を埋め込んでいます。別モジュールの配置は不要です。
機械学習に使う3列は、人口増減率（%）、単独世帯割合の差（ポイント）、年齢判明者中の65歳以上割合の差（ポイント）です。
年齢不詳率は特徴量に混ぜず、各年と差を説明用の品質情報として保持します。


In [ ]:
"""Pure driver-side KMeans logic shared verbatim with nb03_cluster_changes.

Inputs are already validated Spark comparison rows; the checks below independently
protect the ML boundary. No Oracle connection or Spark session is needed to test it.
"""
import hashlib
import itertools
import json
import math
from collections import Counter

import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler

FEATURES = (
    "population_growth_pct",
    "single_share_change_pp",
    "elderly_share_change_pp",
)
SOURCE_COLUMNS = (
    "area_code", "area_name", "population_2020", "population_2025",
    "population_growth_pct", "single_households_pct_2020",
    "single_households_pct_2025", "single_share_change_pp",
    "age_65plus_share_known_pct_2020", "age_65plus_share_known_pct_2025",
    "elderly_share_change_pp", "age_unknown_pct_2020", "age_unknown_pct_2025",
    "age_unknown_change_pp",
)
FEATURE_DEFINITIONS = [
    {"metric": FEATURES[0], "unit": "percent", "period": "2020-2025",
     "definition": "100 * (population_2025 / population_2020 - 1); five-year growth, not annual growth"},
    {"metric": FEATURES[1], "unit": "percentage_points", "period": "2020-2025",
     "definition": "single-household share in 2025 minus 2020; each share denominator is general households"},
    {"metric": FEATURES[2], "unit": "percentage_points", "period": "2020-2025",
     "definition": "65-plus share in 2025 minus 2020; each share denominator is population with known age"},
]


def json_bytes(value):
    return (json.dumps(value, ensure_ascii=False, allow_nan=False,
                       sort_keys=True, indent=2) + "\n").encode("utf-8")


def digest(value):
    return hashlib.sha256(json_bytes(value)).hexdigest()


def comparison_input_hash(records):
    """Match comparison_core.canonical_hash exactly, before projecting columns."""
    data = json.dumps(sorted(records, key=lambda row: row["area_code"]),
                      ensure_ascii=False, sort_keys=True, allow_nan=False,
                      separators=(",", ":")).encode("utf-8")
    return hashlib.sha256(data).hexdigest()


def verify_comparison_input(metadata, raw_records, comparison_run_id, input_path):
    required = {"status": "PASS", "comparison_run_id": comparison_run_id,
                "target": input_path, "period": "2020-2025", "series": "original",
                "unknown_policy": "preserve_dash_as_null"}
    for key, expected in required.items():
        if metadata.get(key) != expected:
            raise ValueError(f"Comparison validation missing or mismatched {key}: expected {expected!r}")
    expected_rows = metadata.get("rows")
    if isinstance(expected_rows, bool) or expected_rows not in (61, 62) or len(raw_records) != expected_rows:
        raise ValueError("Comparison validation row count mismatch")
    # Hash the complete raw Spark row dictionaries. Dropping an extra source
    # column or coercing types before hashing would weaken the manifest check.
    observed_hash = comparison_input_hash(raw_records)
    if metadata.get("comparison_sha256") != observed_hash:
        raise ValueError("Comparison input hash mismatch or missing comparison_sha256")
    return observed_hash


def report_round(value):
    rounded = round(float(value), 2)
    return 0.0 if rounded == 0.0 else rounded


def finite_number(value, name):
    if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value):
        raise ValueError(f"{name}: finite numeric value required")
    return float(value)


def validate_records(records, expected_rows=62):
    if len(records) != expected_rows:
        raise ValueError(f"Expected {expected_rows} municipalities, received {len(records)}")
    clean = []
    codes = set()
    for record in records:
        row = {column: record[column] for column in SOURCE_COLUMNS}
        code = row["area_code"]
        if not isinstance(code, str) or len(code) != 5 or not code.isdigit() or code in codes:
            raise ValueError(f"Invalid or duplicate area_code: {code!r}")
        codes.add(code)
        if not isinstance(row["area_name"], str) or not row["area_name"].strip():
            raise ValueError(f"{code}: nonempty area_name required")
        for column in ("population_2020", "population_2025"):
            value = row[column]
            if isinstance(value, bool) or not isinstance(value, int) or value <= 0:
                raise ValueError(f"{code}.{column}: positive integer required")
        for column in SOURCE_COLUMNS[4:]:
            value = row[column]
            if value is None and column.startswith("age_unknown_"):
                continue
            row[column] = finite_number(value, f"{code}.{column}")
            if column.endswith(("_2020", "_2025")) and not 0 <= row[column] <= 100:
                raise ValueError(f"{code}.{column}: share outside 0..100")
        expected = {
            "population_growth_pct": (row["population_2025"] / row["population_2020"] - 1) * 100,
            "single_share_change_pp": row["single_households_pct_2025"] - row["single_households_pct_2020"],
            "elderly_share_change_pp": row["age_65plus_share_known_pct_2025"] - row["age_65plus_share_known_pct_2020"],
        }
        for column, value in expected.items():
            if not math.isclose(row[column], value, rel_tol=1e-9, abs_tol=1e-8):
                raise ValueError(f"{code}.{column}: delta disagrees with source values")
        unknowns = row["age_unknown_pct_2020"], row["age_unknown_pct_2025"]
        if any(value is None for value in unknowns):
            if row["age_unknown_change_pp"] is not None:
                raise ValueError(f"{code}: unknown-age delta must be NULL when either input is NULL")
        elif row["age_unknown_change_pp"] is None or not math.isclose(
                row["age_unknown_change_pp"], unknowns[1] - unknowns[0], abs_tol=1e-8):
            raise ValueError(f"{code}: unknown-age delta disagrees with source values")
        clean.append(row)
    return sorted(clean, key=lambda row: row["area_code"])


def evaluate_clusters(matrix, seeds=(42, 7, 21), k_values=range(2, 7)):
    """Return candidates and final model; all three seeds must avoid singleton clusters."""
    unique_count = len(np.unique(matrix, axis=0))
    if unique_count < 2:
        raise ValueError("All change vectors are identical; clustering is not meaningful")
    evaluations, summaries, models = [], [], {}
    for k in k_values:
        if k >= len(matrix) or k > unique_count:
            summaries.append({"k": int(k), "eligible": False,
                              "reason": "insufficient distinct vectors",
                              "mean_silhouette": None, "min_cluster_size_all_seeds": None,
                              "min_pairwise_ari": None, "mean_pairwise_ari": None})
            continue
        labels_by_seed = {}
        scores, min_sizes = [], []
        for seed in seeds:
            model = KMeans(n_clusters=k, random_state=seed, n_init=20,
                           max_iter=300, tol=1e-4, algorithm="lloyd").fit(matrix)
            labels = model.labels_
            labels_by_seed[seed] = labels
            models[(k, seed)] = model
            sizes = np.bincount(labels, minlength=k)
            score = float(silhouette_score(matrix, labels))
            scores.append(score)
            min_sizes.append(int(sizes.min()))
            evaluations.append({"k": int(k), "seed": int(seed), "silhouette": score,
                                "inertia": float(model.inertia_), "min_cluster_size": int(sizes.min()),
                                "singleton_count": int(np.sum(sizes == 1)),
                                "cluster_sizes": [int(value) for value in sizes]})
        pairwise = [{"seed_a": int(a), "seed_b": int(b),
                     "ari": float(adjusted_rand_score(labels_by_seed[a], labels_by_seed[b]))}
                    for a, b in itertools.combinations(seeds, 2)]
        for row in evaluations:
            if row["k"] == k:
                row["ari_to_seed42"] = float(adjusted_rand_score(labels_by_seed[42], labels_by_seed[row["seed"]]))
        summaries.append({"k": int(k), "eligible": min(min_sizes) >= 2,
                          "reason": "eligible" if min(min_sizes) >= 2 else "singleton cluster in at least one seed",
                          "mean_silhouette": float(np.mean(scores)),
                          "min_cluster_size_all_seeds": min(min_sizes),
                          "min_pairwise_ari": min(item["ari"] for item in pairwise),
                          "mean_pairwise_ari": float(np.mean([item["ari"] for item in pairwise])),
                          "pairwise_ari": pairwise})
    eligible = [row for row in summaries if row["eligible"]]
    if not eligible:
        raise ValueError("No k from 2..6 avoids singleton clusters for every seed. Review data/features; do not force a result")
    selected = min(eligible, key=lambda row: (-round(row["mean_silhouette"], 12), row["k"]))
    return evaluations, summaries, models[(selected["k"], 42)]


def unknown_context(members):
    output = {}
    for year in (2020, 2025):
        values = [row[f"age_unknown_pct_{year}"] for row in members
                  if row[f"age_unknown_pct_{year}"] is not None]
        output[f"year_{year}"] = {
            "available_count": len(values), "null_count": len(members) - len(values),
            "mean_pct": report_round(np.mean(values)) if values else None,
            "max_pct": report_round(max(values)) if values else None,
        }
    values = [row["age_unknown_change_pp"] for row in members if row["age_unknown_change_pp"] is not None]
    output["change"] = {
        "available_count": len(values), "null_count": len(members) - len(values),
        "mean_pp": report_round(np.mean(values)) if values else None,
        "max_abs_pp": report_round(max(abs(value) for value in values)) if values else None,
    }
    return output


def run_analysis(records, comparison_run_id, ai_run_id, target_area_code="13104", top_n=5,
                 expected_rows=62, coverage=None):
    coverage = coverage or {"input_municipality_count": 62, "excluded_area_codes": [], "excluded_reasons": {}}
    excluded = coverage.get("excluded_area_codes", [])
    if (expected_rows not in (61, 62) or coverage.get("input_municipality_count") != 62
            or len(set(excluded)) != len(excluded) or len(excluded) != 62 - expected_rows):
        raise ValueError("Coverage must account explicitly for all 62 input municipalities")
    if excluded and (excluded != ["13209"] or not coverage.get("excluded_reasons")):
        raise ValueError("This analysis only supports the documented Machida boundary exclusion")
    records = validate_records(records, expected_rows=expected_rows)
    if any(row["area_code"] in excluded for row in records):
        raise ValueError("An excluded municipality remains in the ML input")
    matrix = np.array([[row[feature] for feature in FEATURES] for row in records], dtype=np.float64)
    if np.any(np.var(matrix, axis=0) == 0):
        raise ValueError("A selected feature has zero variance. Review feature design before clustering")
    scaler = StandardScaler().fit(matrix)
    standardized = scaler.transform(matrix)
    evaluations, summaries, model = evaluate_clusters(standardized)
    assignments = []
    for index, row in enumerate(records):
        assignments.append({**row, "cluster_id": int(model.labels_[index]),
                            **{f"z_{feature}": float(standardized[index, column])
                               for column, feature in enumerate(FEATURES)}})
    profiles, payload_clusters = [], []
    for cluster_id in range(model.n_clusters):
        members = [row for row in assignments if row["cluster_id"] == cluster_id]
        profile = {"cluster_id": cluster_id, "member_count": len(members)}
        metrics = {}
        for definition in FEATURE_DEFINITIONS:
            feature = definition["metric"]
            values = [row[feature] for row in members]
            for aggregate, value in (("mean", np.mean(values)), ("min", min(values)), ("max", max(values))):
                profile[f"{feature}_{aggregate}"] = float(value)
            metrics[feature] = {"mean": report_round(profile[f"{feature}_mean"]),
                                "unit": definition["unit"], "period": definition["period"]}
        context = unknown_context(members)
        profile["age_unknown_context_json"] = json.dumps(context, ensure_ascii=False, allow_nan=False, sort_keys=True)
        profiles.append(profile)
        payload_clusters.append({"cluster_id": cluster_id, "member_count": len(members),
                                 "members": [{"area_code": row["area_code"], "area_name": row["area_name"]} for row in members],
                                 "metrics": metrics, "age_unknown": context})
    indices = {row["area_code"]: index for index, row in enumerate(records)}
    if target_area_code not in indices:
        raise ValueError(f"Unknown nearest-neighbor target: {target_area_code}")
    if isinstance(top_n, bool) or not isinstance(top_n, int) or not 1 <= top_n < len(records):
        raise ValueError("top_n must be an integer between 1 and municipality_count-1")
    target_index = indices[target_area_code]
    distances = np.linalg.norm(standardized - standardized[target_index], axis=1)
    order = sorted((index for index in range(len(records)) if index != target_index),
                   key=lambda index: (float(distances[index]), records[index]["area_code"]))[:top_n]
    neighbors = [{"target_area_code": target_area_code, "target_area_name": records[target_index]["area_name"],
                  "rank": rank + 1, "area_code": records[index]["area_code"], "area_name": records[index]["area_name"],
                  "cluster_id": int(model.labels_[index]), "standardized_distance": float(distances[index]),
                  **{feature: records[index][feature] for feature in FEATURES}}
                 for rank, index in enumerate(order)]
    payload = {"schema_version": "1.0", "comparison_run_id": comparison_run_id, "ai_run_id": ai_run_id,
               "period": {"from": 2020, "to": 2025}, "municipality_count": len(records),
               "selected_k": int(model.n_clusters), "aggregation": "unweighted municipality mean",
               "coverage": coverage,
               "rounding": "Cluster metric means and unknown-age context are rounded to 2 decimals. Parquet metrics retain full precision.",
               "age_unknown_context_definition": "Year means use available values in each year; change means use paired available values only. NULL is not treated as zero. Different available subsets can make the paired mean change differ from the difference of year means.",
               "feature_definitions": FEATURE_DEFINITIONS, "clusters": payload_clusters,
               "interpretation_limits": [
                   "Exploratory grouping of observed change vectors, not future prediction or causal inference.",
                   "Cluster IDs are arbitrary labels, not rankings.",
                   "Known-age denominators exclude unknown age; compare unknown-age rates alongside elderly-share change.",
                   "Nearest means similar standardized change pattern, not geographical proximity.",
                   "Each municipality has equal weight; cluster means are not pooled population or household rates.",
               ]}
    model_parameters = {"features": list(FEATURES), "scaler_mean": scaler.mean_.tolist(),
                        "scaler_scale": scaler.scale_.tolist(), "cluster_centers_standardized": model.cluster_centers_.tolist(),
                        "cluster_centers_original_units": scaler.inverse_transform(model.cluster_centers_).tolist(),
                        "selected_k": int(model.n_clusters), "final_seed": 42, "candidate_seeds": [42, 7, 21],
                        "n_init": 20, "max_iter": 300, "tol": 1e-4, "algorithm": "lloyd",
                        "selection": "Maximum mean silhouette among candidates where all seeds have minimum cluster size >=2; tie at 12 decimals chooses lower k"}
    return {"input_records": records, "source_sha256": digest(records), "assignments": assignments,
            "profiles": profiles, "evaluations": evaluations, "k_summary": summaries,
            "nearest_neighbors": neighbors, "profile_payload": payload, "model_parameters": model_parameters}


def create_plots(result, output_directory):
    """English axes and area codes avoid requiring Japanese fonts in the cluster."""
    from pathlib import Path
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from matplotlib.lines import Line2D

    output_directory = Path(output_directory)
    output_directory.mkdir(parents=True, exist_ok=True)
    rows = result["assignments"]
    k = result["model_parameters"]["selected_k"]
    colors = plt.get_cmap("tab10")
    # Label the reference municipality only; the neighbor table supplies the five
    # names without overlapping labels in a dense scatter plot.
    targets = {result["nearest_neighbors"][0]["target_area_code"]}
    label_map = {FEATURES[0]: "Population growth 2020-2025 (%)",
                 FEATURES[1]: "Single-household share change (pp)",
                 FEATURES[2]: "Known-age 65+ share change (pp)"}
    pairs = [(FEATURES[0], FEATURES[1]), (FEATURES[0], FEATURES[2]), (FEATURES[1], FEATURES[2])]
    fig, axes = plt.subplots(1, 3, figsize=(18, 5.5), constrained_layout=True)
    for axis, (x, y) in zip(axes, pairs):
        for cluster_id in range(k):
            members = [row for row in rows if row["cluster_id"] == cluster_id]
            axis.scatter([row[x] for row in members], [row[y] for row in members],
                         color=colors(cluster_id), label=f"Cluster {cluster_id}", alpha=0.8, s=35)
        for row in rows:
            if row["area_code"] in targets:
                axis.annotate(row["area_code"], (row[x], row[y]), xytext=(4, 4), textcoords="offset points", fontsize=7)
        axis.axhline(0, color="#888888", linewidth=0.6)
        axis.axvline(0, color="#888888", linewidth=0.6)
        axis.set_xlabel(label_map[x])
        axis.set_ylabel(label_map[y])
        axis.grid(alpha=0.15)
    axes[-1].legend(fontsize=8)
    fig.suptitle(f"Observed municipality changes | n={len(rows)}, k={k} | unweighted municipalities")
    first = output_directory / "feature_changes.png"
    fig.savefig(first, dpi=160)
    plt.close(fig)

    fig, axis = plt.subplots(figsize=(10, 8), constrained_layout=True)
    for row in rows:
        x0, x1 = row["age_65plus_share_known_pct_2020"], row["age_65plus_share_known_pct_2025"]
        y0, y1 = row["single_households_pct_2020"], row["single_households_pct_2025"]
        color = colors(row["cluster_id"])
        axis.annotate("", xy=(x1, y1), xytext=(x0, y0),
                      arrowprops={"arrowstyle": "->", "color": color, "alpha": 0.6, "lw": 1.2})
        axis.scatter([x0], [y0], facecolors="none", edgecolors=[color], s=18, alpha=0.7)
        axis.scatter([x1], [y1], color=[color], s=16, alpha=0.8)
        if row["area_code"] in targets:
            axis.annotate(row["area_code"], (x1, y1), xytext=(3, 3), textcoords="offset points", fontsize=8)
    handles = [Line2D([0], [0], color=colors(i), marker="o", linestyle="", label=f"Change cluster {i}") for i in range(k)]
    axis.legend(handles=handles, fontsize=8)
    axis.set_xlabel("Known-age 65+ share (%)")
    axis.set_ylabel("Single-household share of general households (%)")
    axis.set_title(f"2020 (open) to 2025 (filled) | n={len(rows)}\nColor comes from the three-feature change model")
    axis.grid(alpha=0.2)
    second = output_directory / "share_movements.png"
    fig.savefig(second, dpi=160)
    plt.close(fig)

    fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
    summaries = [row for row in result["k_summary"] if row["mean_silhouette"] is not None]
    for row in summaries:
        color = "#087e8b" if row["eligible"] else "#bbbbbb"
        axes[0].scatter(row["k"], row["mean_silhouette"], color=color, s=65)
        axes[1].scatter(row["k"], row["min_pairwise_ari"], color=color, s=65)
    for axis in axes:
        axis.axvline(k, color="#c04b2e", linestyle="--", label=f"Selected k={k}")
        axis.set_xlabel("Number of clusters (k)")
        axis.set_xticks(range(2, 7))
        axis.grid(alpha=0.2)
    axes[0].set_ylabel("Mean silhouette across three seeds")
    axes[1].set_ylabel("Minimum adjusted Rand index across seed pairs")
    axes[0].legend()
    fig.suptitle("Teal: all seeds have minimum cluster size >=2; gray: excluded candidate")
    third = output_directory / "k_selection.png"
    fig.savefig(third, dpi=160)
    plt.close(fig)
    return [str(first), str(second), str(third)]


In [ ]:
with open(f"{COMPARISON_ROOT}/validation.json", encoding="utf-8") as handle:
    comparison_validation = json.load(handle)
EXPECTED_ROWS = comparison_validation.get("rows")
coverage = {
    "input_municipality_count": comparison_validation.get("input_municipality_count", 62),
    "excluded_area_codes": comparison_validation.get("excluded_area_codes", []),
    "excluded_reasons": comparison_validation.get("excluded_reasons", {}),
}
input_df = spark.read.parquet(INPUT_PATH)
missing_columns = sorted(set(SOURCE_COLUMNS) - set(input_df.columns))
if missing_columns:
    raise ValueError(f"入力列が不足しています: {missing_columns}")
# Driverに取るのは最大63行。列の選択・型変換前の全列でnb02のハッシュと照合します。
source_rows = [row.asDict(recursive=True) for row in input_df.orderBy("area_code").limit(63).collect()]
verify_comparison_input(comparison_validation, source_rows, COMPARISON_RUN_ID, INPUT_PATH)
result = run_analysis(source_rows, COMPARISON_RUN_ID, AI_RUN_ID,
                      target_area_code=TARGET_AREA_CODE, top_n=TOP_N,
                      expected_rows=EXPECTED_ROWS, coverage=coverage)
print(json.dumps({"municipality_count": len(source_rows), "coverage": coverage,
                  "k_summary": result["k_summary"], "nearest_neighbors": result["nearest_neighbors"]},
                 ensure_ascii=False, allow_nan=False, indent=2))


## 結果を保存し、読み戻して検査する

SparkのParquetには丸め前の数値を保存します。生成AI向けJSONの平均値は小数第2位までです。
最終 `validation.json` は、保存先を読み戻して内容一致を検査した後に書き込みます。
失敗したrunの途中成果物は残します。修正後は新しい `ai_run_id` を使ってください。


In [ ]:
from pyspark.sql.types import (StructType, StructField, StringType, LongType,
                              DoubleType, BooleanType, ArrayType)
import sklearn
import scipy

def schema_from_columns(columns):
    fields = []
    for name in columns:
        if name in ("area_code", "area_name", "target_area_code", "target_area_name", "age_unknown_context_json", "reason"):
            kind = StringType()
        elif name in ("population_2020", "population_2025", "cluster_id", "member_count", "k", "seed", "rank", "min_cluster_size", "singleton_count", "min_cluster_size_all_seeds"):
            kind = LongType()
        elif name == "eligible":
            kind = BooleanType()
        elif name == "cluster_sizes":
            kind = ArrayType(LongType(), containsNull=False)
        else:
            kind = DoubleType()
        nullable = name.startswith("age_unknown_") or name in ("mean_silhouette", "min_cluster_size_all_seeds", "min_pairwise_ari", "mean_pairwise_ari")
        fields.append(StructField(name, kind, nullable=nullable))
    return StructType(fields)

OUTPUT_ROOT.mkdir(parents=True, exist_ok=False)

def write_json_new(name, value):
    data = json_bytes(value)
    with (OUTPUT_ROOT / name).open("xb") as handle:
        handle.write(data)
    if (OUTPUT_ROOT / name).read_bytes() != data:
        raise IOError(f"JSON read-back mismatch: {name}")
    return hashlib.sha256(data).hexdigest()

def write_verified_parquet(name, rows, sort_keys):
    columns = list(rows[0])
    schema = schema_from_columns(columns)
    path = str(OUTPUT_ROOT / name)
    spark.createDataFrame(rows, schema=schema).write.mode("errorifexists").parquet(path)
    restored = [row.asDict(recursive=True) for row in spark.read.parquet(path).select(*columns).limit(len(rows) + 1).collect()]
    key = lambda row: tuple(row[column] for column in sort_keys)
    if json_bytes(sorted(rows, key=key)) != json_bytes(sorted(restored, key=key)):
        raise ValueError(f"Parquet read-back mismatch: {name}")
    return {"rows": len(rows), "sha256": digest(sorted(rows, key=key)), "path": path}

outputs = {}
outputs["input_snapshot"] = write_verified_parquet("input_snapshot", result["input_records"], ["area_code"])
outputs["assignments"] = write_verified_parquet("assignments", result["assignments"], ["area_code"])
outputs["profiles"] = write_verified_parquet("profiles", result["profiles"], ["cluster_id"])
outputs["evaluations"] = write_verified_parquet("evaluations", result["evaluations"], ["k", "seed"])
summary_rows = [{key: value for key, value in row.items() if key != "pairwise_ari"} for row in result["k_summary"]]
outputs["k_summary"] = write_verified_parquet("k_summary", summary_rows, ["k"])
outputs["nearest_neighbors"] = write_verified_parquet("nearest_neighbors", result["nearest_neighbors"], ["rank"])
write_json_new("model_parameters.json", result["model_parameters"])
write_json_new("k_evaluation.json", {"evaluations": result["evaluations"], "k_summary": result["k_summary"]})
write_json_new("comparison_validation_snapshot.json", comparison_validation)
payload_sha256 = write_json_new("profile_payload.json", result["profile_payload"])
chart_paths = create_plots(result, OUTPUT_ROOT / "charts")
for chart_path in chart_paths:
    if Path(chart_path).stat().st_size == 0:
        raise ValueError(f"Empty plot: {chart_path}")

run_metadata = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "python_version": platform.python_version(),
    "versions": {"numpy": np.__version__, "scikit_learn": sklearn.__version__,
                 "scipy": scipy.__version__, "spark": spark.version},
    "comparison_run_id": COMPARISON_RUN_ID, "ai_run_id": AI_RUN_ID,
    "source_sha256": result["source_sha256"], "source_path": INPUT_PATH,
    "comparison_sha256": comparison_validation["comparison_sha256"],
    "coverage": coverage, "model_parameters": result["model_parameters"],
    "outputs": outputs, "charts": chart_paths,
}
write_json_new("run_metadata.json", run_metadata)
validation = {
    "status": "PASS", "ai_run_id": AI_RUN_ID, "comparison_run_id": COMPARISON_RUN_ID,
    "rows": len(source_rows), "municipality_count": len(source_rows),
    "selected_k": result["model_parameters"]["selected_k"],
    "source_sha256": result["source_sha256"], "profile_payload_sha256": payload_sha256,
    "comparison_sha256": comparison_validation["comparison_sha256"],
    "output_root": str(OUTPUT_ROOT), "coverage": coverage,
    "python_version": run_metadata["python_version"], "versions": run_metadata["versions"],
    "checks": {"input": "PASS", "parquet_read_back": "PASS", "json_read_back": "PASS"},
    "interpretation_review": "REQUIRED",
}
# PASSマーカーはすべての出力・読戻し検査の最後に作成します。
write_json_new("validation.json", validation)
print(json.dumps(validation, ensure_ascii=False, allow_nan=False, indent=2))


## 図を表示して、解釈を確認する

生成AIへ進む前に候補kのsilhouette、seed間ARI、クラスタ人数を確認してください。
最小人数条件を満たすことや数値検査PASSは、分類の社会的な妥当性を保証しません。
町田市を除外した場合は、図や記事の集計対象を「61市区町村」と記載し、62市区町村の結果と書かないでください。
矢印図の割合の分母は、単独世帯＝一般世帯、65歳以上＝年齢判明者です。
新宿区と似た変化でも、人口規模、現在の割合、地理的近さが似ていることは意味しません。


In [ ]:
from IPython.display import Image, display
for path in chart_paths:
    display(Image(filename=path))
